# 02 · Features

We turn the clean dataset from notebook 01 into model-ready features, using only information available **before or during a film's release week**.

| Group | Features | Source |
|---|---|---|
| Structured | budget, runtime, release date, genres, language, country, counts, sequel flag | TMDB fields |
| Track record | how experienced each film's cast and key crew are, and how well their **earlier** films did | cast and crew lists |
| Plot | sentence embedding of the plot summary | `overview` |
| Keywords | sentence embedding of the keyword list | `Keywords` |

The last three groups process the cast, crew and text fields; notebook 03 compares models with and without them.

**Output:** `features.parquet` (one row per film, keyed on `imdb_id`, with the target and the split).

1. Imports
2. Load
3. Structured features
4. Sequels
5. Cast and crew track record
6. Text embeddings: plot and keywords
7. Assemble and save

## 1. Imports

In [1]:
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
sns.set_theme(style="whitegrid", context="notebook")

## 2. Load

`split` comes from notebook 01. Anything we *learn* from the data in this notebook (most common categories, the dimension reduction of the embeddings) is fitted on the **train** films only, so the validation and test films stay unseen.

In [2]:
df = pd.read_parquet("movies_clean.parquet")
is_train = df["split"] == "train"
print(df.shape)
df["split"].value_counts()

(6090, 26)


split
train    4263
test      914
val       913
Name: count, dtype: int64

In [3]:
def parse(s):
    """TMDB list columns are stored as text such as "[{'name': 'Comedy'}]"."""
    if not isinstance(s, str):
        return []
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []


for col in ["genres", "production_companies", "production_countries", "spoken_languages",
            "belongs_to_collection", "Keywords", "cast", "crew"]:
    df[col] = df[col].map(parse)

## 3. Structured features

Missing values (e.g. an unknown budget, 26% of films) stay as `NaN`.

In [4]:
F = pd.DataFrame(index=df.index)

F["log_budget_cpi"] = df["log_budget_cpi"]
F["runtime"] = df["runtime"].replace(0, np.nan)
F["year"] = df["year"]
F["month"] = df["release_date"].dt.month
F["weekday"] = df["release_date"].dt.dayofweek
F["n_genres"] = df["genres"].str.len()
F["n_production_companies"] = df["production_companies"].str.len()
F["n_countries"] = df["production_countries"].str.len()
F["n_languages"] = df["spoken_languages"].str.len()

F.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
log_budget_cpi,4489.0,17.29,1.35,10.06,16.61,17.49,18.23,20.11
runtime,6088.0,107.43,18.59,46.00,95.00,104.00,117.00,254.00
year,6090.0,2002.79,9.91,1980.00,1995.00,2005.00,2011.00,2017.00
month,6090.0,6.84,3.35,1.00,4.00,7.00,10.00,12.00
weekday,6090.0,3.29,1.29,0.00,2.00,4.00,4.00,6.00
n_genres,6090.0,2.52,1.12,0.00,2.00,2.00,3.00,8.00
n_production_companies,6090.0,2.95,2.26,0.00,1.00,2.00,4.00,26.00
n_countries,6090.0,1.37,0.83,0.00,1.00,1.00,2.00,12.00
n_languages,6090.0,1.46,0.90,0.00,1.00,1.00,2.00,9.00


For original language and main production country, the 8 most common values among the training films each get a 0/1 column; all other values are grouped as "other". Each genre gets its own 0/1 column, since a film can have several.

In [5]:
def one_hot_top(values, prefix, k=8):
    top = values[is_train].value_counts().index[:k]
    return pd.get_dummies(values.where(values.isin(top), "other"), prefix=prefix, dtype=int)


country_main = df["production_countries"].map(lambda L: L[0]["iso_3166_1"] if L else "none")
cat = pd.concat([one_hot_top(df["original_language"], "lang"),
                 one_hot_top(country_main, "country")], axis=1)

genre_names = df["genres"].map(lambda L: [g["name"] for g in L])
genre = (pd.get_dummies(genre_names.explode()).groupby(level=0).max()
           .reindex(df.index, fill_value=0).astype(int))
genre.columns = ["genre_" + c.replace(" ", "_") for c in genre.columns]

print(f"language / country columns: {cat.shape[1]}   genre columns: {genre.shape[1]}")

language / country columns: 18   genre columns: 19


In [6]:
cat.head()

,lang_cn,lang_en,lang_es,lang_fr,lang_hi,lang_ja,lang_ko,lang_other,lang_zh,country_AU,country_CA,country_DE,country_FR,country_GB,country_IN,country_US,country_none,country_other
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [7]:
genre.head()

,genre_Action,genre_Adventure,genre_Animation,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Family,genre_Fantasy,genre_Foreign,genre_History,genre_Horror,genre_Music,genre_Mystery,genre_Romance,genre_Science_Fiction,genre_Thriller,genre_War,genre_Western
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0


## 4. Sequels

TMDB marks every film in a franchise as part of a collection, including the first, which only joined once a sequel was made, i.e. after it succeeded. We therefore mark a film as a sequel only if another film of the same collection was released **before** it. When earlier films are not in our data, we fall back on the title (a number or "Part", e.g. *Hot Tub Time Machine 2*).

In [8]:
collection = df["belongs_to_collection"].map(lambda L: L[0]["name"] if L else None)
first_date = df.groupby(collection)["release_date"].transform("min")
later_in_collection = collection.notna() & (df["release_date"] > first_date)
sequel_title = df["title"].str.contains(r"\b(?:[2-9]|II|III|IV|V|VI|Part)\b", regex=True, na=False)

F["is_sequel"] = (later_in_collection | (collection.notna() & sequel_title)).astype(int)
pd.crosstab(collection.notna().rename("in a collection"), F["is_sequel"])

is_sequel,0,1
in a collection,,
False,4854,0
True,602,634


## 5. Cast and crew track record

For each film and each role we describe the people involved by their record **before** this film's release:

- `exp_<role>`: how many of them had at least 2 earlier films in our data (experience).
- `prev_rev_<role>`: the average ln revenue (2025 dollars) of their earlier films, over people with at least one earlier film. `NaN` if nobody has one.

Only films released strictly before the film count, so no feature uses information from the future. Revenue history comes from train and validation films only; **test revenues are never used**.

| Role | Who |
|---|---|
| actor | the 10 top-billed cast members |
| director | job "Director" |
| producer | jobs "Producer" and "Executive Producer" |
| writer | Writing department |
| composer | job "Original Music Composer" |
| cinematographer | job "Director of Photography" |
| editor | job "Editor" |

In [9]:
role_people = {
    "actor": df["cast"].map(lambda L: [p["name"] for p in L if p["order"] < 10]),
    "director": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Director"]),
    "producer": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] in ("Producer", "Executive Producer")]),
    "writer": df["crew"].map(lambda L: [p["name"] for p in L if p["department"] == "Writing"]),
    "composer": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Original Music Composer"]),
    "cinematographer": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Director of Photography"]),
    "editor": df["crew"].map(lambda L: [p["name"] for p in L if p["job"] == "Editor"]),
}

MIN_PRIOR = 2
known_revenue = df["log_revenue_cpi"].where(df["split"] != "test")   # test revenues hidden


def track_record(people):
    """Per film: number of experienced people, and mean ln revenue of their earlier films."""
    long = (people.map(lambda L: list(dict.fromkeys(L)))          # each person once per film
                  .explode().dropna().rename("person").to_frame())
    long["film"] = long.index
    long["date"] = df.loc[long.index, "release_date"].values
    long["rev"] = known_revenue.loc[long.index].values

    # Totals per person and release date, then running totals over strictly earlier dates
    per_date = (long.groupby(["person", "date"])
                    .agg(n=("film", "size"), rev_sum=("rev", "sum"), rev_cnt=("rev", "count"))
                    .reset_index())
    g = per_date.groupby("person")
    per_date["n_prior"] = g["n"].cumsum() - per_date["n"]
    per_date["rev_sum_prior"] = g["rev_sum"].cumsum() - per_date["rev_sum"]
    per_date["rev_cnt_prior"] = g["rev_cnt"].cumsum() - per_date["rev_cnt"]

    long = long.merge(per_date[["person", "date", "n_prior", "rev_sum_prior", "rev_cnt_prior"]],
                      on=["person", "date"], how="left")
    long["prev_rev"] = (long["rev_sum_prior"] / long["rev_cnt_prior"]).where(long["rev_cnt_prior"] > 0)

    by_film = long.groupby("film")
    out = pd.DataFrame({
        "exp": by_film["n_prior"].apply(lambda s: (s >= MIN_PRIOR).sum()),
        "prev_rev": by_film["prev_rev"].mean(),
    })
    return out.reindex(df.index).fillna({"exp": 0})


track = pd.DataFrame(index=df.index)
for role, people in role_people.items():
    t = track_record(people)
    track[f"exp_{role}"] = t["exp"].astype(int)
    track[f"prev_rev_{role}"] = t["prev_rev"]

track.describe().T[["count", "mean", "50%", "max"]].round(2)

,count,mean,50%,max
exp_actor,6090.0,4.35,4.00,11.00
prev_rev_actor,5667.0,17.67,17.77,21.76
exp_director,6090.0,0.39,0.00,6.00
prev_rev_director,3221.0,17.82,18.12,21.49
exp_producer,6090.0,1.57,1.00,17.00
prev_rev_producer,4040.0,17.96,18.21,21.49
exp_writer,6090.0,0.57,0.00,8.00
prev_rev_writer,3281.0,17.98,18.23,21.70
exp_composer,6090.0,0.39,0.00,3.00
prev_rev_composer,2639.0,18.12,18.37,21.49


A quick check that the revenue history carries signal: correlation of each `prev_rev_<role>` with the film's own ln revenue, on training films.

In [10]:
track.loc[is_train, [c for c in track if c.startswith("prev_rev")]] \
     .corrwith(df.loc[is_train, "log_revenue_cpi"]).round(2).rename("correlation with ln revenue")

prev_rev_actor              0.28
prev_rev_director           0.47
prev_rev_producer           0.53
prev_rev_writer             0.43
prev_rev_composer           0.37
prev_rev_cinematographer    0.38
prev_rev_editor             0.43
Name: correlation with ln revenue, dtype: float64

## 6. Text embeddings: plot and keywords

We encode each film's plot summary (`overview`) and its keyword list with a pretrained sentence-embedding model, **all-MiniLM-L6-v2** (Sentence-BERT). It maps a text to a 384-dimensional vector so that texts with similar meaning get similar vectors. The model is pretrained on general text and does not see our revenues.

A few TMDB keywords are added by users after release and describe reception rather than content; we remove them before encoding.

The model is downloaded on first use (about 90 MB).

In [11]:
POST_RELEASE_KEYWORDS = {"duringcreditsstinger", "aftercreditsstinger", "box office flop",
                         "cult film", "flop", "blockbuster", "oscar winner"}

overview_text = df["overview"].fillna("")
keyword_text = df["Keywords"].map(
    lambda L: ", ".join(k["name"] for k in L if k["name"].lower() not in POST_RELEASE_KEYWORDS))

print(f"films with no overview: {(overview_text == '').sum()}   with no keywords: {(keyword_text == '').sum()}")
keyword_text.head()

films with no overview: 4   with no keywords: 438


0                                   time travel, sequel, hot tub
1                    coronation, duty, marriage, falling in love
2    jazz, obsession, conservatory, music teacher, new york c...
3                                                               
4                                                               
Name: Keywords, dtype: str

In [13]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
TEXT_METHOD = "Sentence-BERT (all-MiniLM-L6-v2)"


def embed(texts):
    return encoder.encode(texts.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)


print("Text method:", TEXT_METHOD)
overview_vec = embed(overview_text)
keyword_vec = embed(keyword_text)
print(overview_vec.shape, keyword_vec.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\zhang\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zhang\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Text method: Sentence-BERT (all-MiniLM-L6-v2)


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

(6090, 384) (6090, 384)


384 dimensions is a lot for about 4,000 training films, so we reduce each embedding to **32 dimensions** with principal component analysis (PCA), fitted on the training films. Films with no text get `NaN`.

In [14]:
N_COMPONENTS = 32


def reduce(vectors, empty, prefix):
    pca = PCA(n_components=N_COMPONENTS, random_state=42).fit(vectors[(is_train & ~empty).values])
    out = pd.DataFrame(pca.transform(vectors), index=df.index,
                       columns=[f"{prefix}_{i}" for i in range(N_COMPONENTS)])
    out[empty] = np.nan
    print(f"{prefix}: {pca.explained_variance_ratio_.sum():.0%} of the variance kept")
    return out


plot_emb = reduce(overview_vec, overview_text == "", "plot_emb")
kw_emb = reduce(keyword_vec, keyword_text == "", "kw_emb")

plot_emb: 42% of the variance kept
kw_emb: 55% of the variance kept


## 7. Assemble and save

One row per film: identifiers, split, target, then the feature groups. The column-name prefixes identify the groups in notebook 03.

In [15]:
features = pd.concat([
    df[["imdb_id", "title", "split", "log_revenue_cpi", "revenue_cpi"]],
    F, cat, genre, track, plot_emb, kw_emb,
], axis=1)

groups = {
    "structured": list(F.columns) + list(cat.columns) + list(genre.columns),
    "track record": list(track.columns),
    "plot embedding": list(plot_emb.columns),
    "keyword embedding": list(kw_emb.columns),
}
print(f"features.parquet: {features.shape}   text method: {TEXT_METHOD}")
pd.Series({g: len(c) for g, c in groups.items()}, name="columns")

features.parquet: (6090, 130)   text method: Sentence-BERT (all-MiniLM-L6-v2)


structured           47
track record         14
plot embedding       32
keyword embedding    32
Name: columns, dtype: int64

In [16]:
features.attrs["text_method"] = TEXT_METHOD
features.to_parquet("features.parquet", index=False)